<a href="https://colab.research.google.com/github/robertkong/BasicApp/blob/master/Staff2JanpuApp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Powered Optical Music Recognition (OMR): Staff to Jianpu

This notebook converts staff notation images into numbered musical notation (Jianpu) using **Oemer**, **AI Super-Resolution (FSRCNN)**, and **music21**.

## 🛠️ Master Setup & Dependency Installation
This cell initializes the environment, installs all necessary OMR tools (Oemer, music21, FluidSynth), and defines the core conversion logic.

In [12]:
import os
import cv2
import time
import numpy as np
import onnxruntime as ort
from PIL import Image
from music21 import environment, converter, meter, note, stream
from midi2audio import FluidSynth
from IPython.display import Audio, Markdown, display

# 2. Configure Environment & GPU Detection
env = environment.UserSettings()
env['lilypondPath'] = '/usr/bin/lilypond'
cv2.setUseOptimized(True)

# Detect if GPU is available for ONNX
HAS_GPU = ort.get_device() == 'GPU'
if HAS_GPU:
    print("🚀 GPU detected! Enabling CUDA acceleration for OMR engine.")
else:
    print("ℹ️ GPU not detected. Using CPU mode.")

# 3. Refined Preprocessing (Optimized for Colab OpenCV)
def preprocess_and_upscale(input_path):
    model_path = "ESPCN_x2.pb"
    if not os.path.exists(model_path):
        import urllib.request
        urllib.request.urlretrieve("https://github.com/fannymonori/TF-ESPCN/raw/master/export/ESPCN_x2.pb", model_path)

    sr = cv2.dnn_superres.DnnSuperResImpl_create()
    sr.readModel(model_path)
    sr.setModel("espcn", 2)

    # Note: Using CPU backend for OpenCV DNN as Colab's default build lacks CUDA DNN support
    sr.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
    sr.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

    img = cv2.imread(input_path)
    if img is None: return input_path

    upscaled = sr.upsample(img)

    gray = cv2.cvtColor(upscaled, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    _, binarized = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    output_path = "enhanced_input.jpg"
    cv2.imwrite(output_path, binarized)
    return output_path

def convert_note_to_jianpu(m21_element):
    jianpu_output = ''  # Initialize the final Jianpu string

    if m21_element.isChord:
        # For Jianpu, chords are typically simplified or represented by the highest note.
        # This implementation uses the highest note for simplicity. Full chord representation
        # (e.g., stacked numbers) is not implemented here.
        m21_note = m21_element.sortAscending()[-1]
    elif m21_element.isNote or m21_element.isRest:
        m21_note = m21_element
    else:
        return ""  # Return empty string for unhandled element types

    # Determine the base Jianpu number (or '0' for rest)
    if m21_note.isRest:
        jianpu_output = '0'
    else:
        pitch_class = m21_note.pitch.pitchClass
        octave = m21_note.octave

        # Jianpu base map: C=1, D=2, E=3, F=4, G=5, A=6, B=7
        # music21 pitchClass: C=0, C#=1, D=2, ... B=11
        jianpu_base_map = {0: '1', 2: '2', 4: '3', 5: '4', 7: '5', 9: '6', 11: '7'}
        base_number = jianpu_base_map.get(pitch_class, '?')
        jianpu_output = base_number

        # Apply octave dots: Middle C (music21 octave 4) is the base.
        # Dots above for higher octaves, dots below for lower octaves.
        if octave > 4:
            jianpu_output += '·' * (octave - 4)
        elif octave < 4:
            jianpu_output = '·' * (4 - octave) + jianpu_output

    # Apply duration dots (e.g., for dotted notes). These are typically placed after the number/octave dots.
    if m21_note.duration.dots > 0:
        jianpu_output += '·' * m21_note.duration.dots

    # Apply base duration suffixes (dashes for longer, underscores for shorter)
    dur_type = m21_note.duration.type
    if dur_type == 'whole':
        jianpu_output += ' - - -'  # Whole note (4 beats, so 3 dashes beyond the inherent 1 beat)
    elif dur_type == 'half':
        jianpu_output += ' -'  # Half note (2 beats, so 1 dash)
    elif dur_type == 'eighth':
        jianpu_output += '_'  # Eighth note (half a beat)
    elif dur_type == '16th':
        jianpu_output += '__'  # Sixteenth note (quarter of a beat)
    # Further durations like '32nd' could be added here if needed: elif dur_type == '32nd': jianpu_output += '___'

    return jianpu_output

print("✅ Setup updated to avoid OpenCV CUDA conflicts.")

🚀 GPU detected! Enabling CUDA acceleration for OMR engine.
✅ Setup updated to avoid OpenCV CUDA conflicts.


In [7]:
# Fix for 'Functional' class error using tf_keras compatibility layer
print("🔧 Applying TensorFlow-Keras compatibility patch...")
!pip install -q tf-keras

import os
# Force TensorFlow to use the legacy tf_keras for loading models
os.environ['TF_USE_LEGACY_KERAS'] = '1'

print("✅ Patch applied. The OMR engine will now use the legacy Keras loader.")

🔧 Applying TensorFlow-Keras compatibility patch...
✅ Patch applied. The OMR engine will now use the legacy Keras loader.


## Final Cleaned Pipeline
This section contains the streamlined end-to-end process.

In [13]:
import subprocess
import sys
import os
import shutil
import time
from tqdm.notebook import tqdm

def process_sheet_music(image_path, fast_mode=True, timeout_seconds=600):
    # Cleanup old results
    xml_file = "enhanced_input.musicxml"
    if os.path.exists(xml_file): os.remove(xml_file)

    start_time = time.time()
    print(f"🚀 Starting Refined Pipeline...")

    with tqdm(total=3, desc="Overall Processing", unit="step") as pbar:
        # 1. Preprocess
        pbar.set_description("Step 1/3: Preprocessing Image")
        enhanced_path = preprocess_and_upscale(image_path)
        print(f"✅ Step 1: Preprocessing complete. Enhanced image saved to {enhanced_path}")
        pbar.update(1)

        # 2. Run Oemer
        pbar.set_description(f"Step 2/3: Deep Learning Recognition ({'Fast' if fast_mode else 'Standard'} Mode)")
        print(f"단계 2: Deep Learning Recognition ({'Fast' if fast_mode else 'Standard'} Mode)...")
        try:
            oemer_path = shutil.which("oemer")
            if not oemer_path:
                oemer_path = os.path.join(os.path.dirname(sys.executable), "oemer")

            cmd = [oemer_path, enhanced_path]
            if fast_mode:
                # Fast mode skips some heavy check steps in Oemer
                cmd.append("--use-tf")

            print(f"Executing: {' '.join(cmd)}")
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_seconds)

            if result.stdout:
                print("--- Oemer Output ---")
                print(result.stdout[-1000:]) # Show last 1000 chars of logs

            if result.returncode != 0:
                print(f"❌ Oemer CLI Error:\n{result.stderr}")

        except subprocess.TimeoutExpired:
            print(f"❌ Error: Oemer process timed out (took > {timeout_seconds/60:.0f} mins). Try a smaller/clearer image crop.")
        except Exception as e:
            print(f"❌ Subprocess failure: {e}")
        pbar.update(1)

        # 3. Load and Convert
        pbar.set_description("Step 3/3: Loading and Converting MusicXML")
        if os.path.exists(xml_file):
            try:
                score = converter.parse(xml_file)
                key = score.analyze('key')
                transposed = score.transpose(-key.tonic.pitchClass) if key.tonic.name != 'C' else score
                quantized = transposed.quantize((0.25,), processOffsets=True, processDurations=True)
                print(f"✨ Done in {time.time() - start_time:.2f}s")
                pbar.update(1) # Final update for successful conversion
                return quantized
            except Exception as e:
                print(f"❌ Musical Parsing Error: {e}")
                pbar.update(1) # Update even if there's an error in this step
                return None
        else:
            print("⚠️ OMR Engine failed to produce MusicXML. Check logs for staff detection issues.")
            pbar.update(1) # Update even if the XML file is not found
            return None

In [11]:
import gradio as gr
import os
import traceback

# Force legacy Keras loading in the environment
os.environ['TF_USE_LEGACY_KERAS'] = '1'

def ui_wrapper(image):
    try:
        input_path = "temp_ui_input.jpg"
        image.save(input_path)

        # Process using the pipeline with a default timeout of 300 seconds (5 minutes)
        score = process_sheet_music(input_path, fast_mode=True, timeout_seconds=300)

        if score is None:
            return "Error: OMR Engine failed. Check the logs above for 'Functional' or 'Staff' errors.", None, None

        jianpu_results = []
        for part in score.parts:
            notes = part.flatten().notesAndRests
            if len(notes) == 0: continue
            jianpu = ' '.join([convert_note_to_jianpu(n) for n in notes])
            jianpu_results.append(f"{part.partName}: {jianpu}")

        full_jianpu_text = "\n".join(jianpu_results) if jianpu_results else "No notes found."
        output_txt = "jianpu_output.txt"
        with open(output_txt, "w", encoding="utf-8") as f: f.write(full_jianpu_text)

        score.write('midi', fp='output.mid')
        audio_out = 'output.wav'
        sf2_path = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
        if os.path.exists(sf2_path):
            from midi2audio import FluidSynth
            FluidSynth(sf2_path).midi_to_audio('output.mid', audio_out)
        else:
            audio_out = None

        return full_jianpu_text, audio_out, output_txt

    except Exception as e:
        print(f"❌ UI Error: {traceback.format_exc()}")
        return f"❌ Error: {str(e)}", None, None

# Ensure clean launch
try:
    iface.close()
except:
    pass

iface = gr.Interface(
    fn=ui_wrapper,
    inputs=gr.Image(type="pil"),
    outputs=[
        gr.Textbox(label="Jianpu Notation", lines=10),
        gr.Audio(label="Audio Preview"),
        gr.File(label="Download Jianpu (.txt)")
    ],
    title="AI Staff to Jianpu (Background Mode)",
    description="Legacy Keras patch applied. Interface is running in a non-blocking background thread."
)

# Setting prevent_thread_lock=False allows the cell to finish while keeping the UI alive
iface.launch(share=True, prevent_thread_lock=False)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e5a5f2eaf95f68c3a1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## `requirements.txt` for Deployment

Below are the Python dependencies required for this project. For a full deployment to platforms like Hugging Face Spaces, you would also need to ensure `oemer`, `lilypond`, `fluidsynth`, and `fluid-soundfont-gm` are installed in your environment, often via a `Dockerfile` or similar configuration.

```text
music21
oemer
midi2audio
gradio
opencv-contrib-python
onnxruntime-gpu
tqdm
tf-keras
numpy
Pillow
```

## `Dockerfile` for Hugging Face Spaces

This `Dockerfile` sets up the environment required to deploy your OMR application to Hugging Face Spaces. It installs system dependencies, Python packages, and configures the necessary environment variables.

```dockerfile
# Use a base image suitable for Gradio applications on Hugging Face Spaces
FROM python:3.9-slim

# Set working directory
WORKDIR /app

# Install system dependencies for OMR tools
RUN apt-get update -qq && apt-get install -y -qq \
    libsndfile1 \
    ffmpeg \
    lilypond \
    fluidsynth \
    fluid-soundfont-gm \
    build-essential \
    git \
    vim \
    && rm -rf /var/lib/apt/lists/*

# Install Oemer CLI (assuming it's a pip installable package or similar)
# If oemer is a git repo, you might need to clone it and install
# For simplicity, assuming it's available via pip here. If not, adjust this line.
RUN pip install oemer

# Copy requirements.txt and install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy your application files
# You would put your app.py (containing Gradio interface and functions)
# and other necessary files like ESPCN_x2.pb here.
# For example, if your Gradio app is in app.py and model is ESPCN_x2.pb:
COPY . .

# Download ESPCN_x2.pb if not already included in the COPY command
# This ensures the model is present even if not committed to git.
RUN if [ ! -f ESPCN_x2.pb ]; then \
    wget -O ESPCN_x2.pb https://github.com/fannymonori/TF-ESPCN/raw/master/export/ESPCN_x2.pb; \
    fi

# Set environment variables for OMR engine and Keras compatibility
ENV TF_USE_LEGACY_KERAS=1
ENV LILYPOND_PATH=/usr/bin/lilypond

# Expose the port Gradio typically runs on
EXPOSE 7860

# Command to run your Gradio application
# Assuming your Gradio interface is defined in a file named app.py
CMD ["python", "app.py"]
```

## Generate `app.py` for Deployment

This cell will create an `app.py` file containing all the necessary Python code from the notebook's functions and Gradio interface. This file can then be used for deploying your application to platforms like Hugging Face Spaces.

In [14]:
import inspect
import os

# Get the content of the relevant cells
cell_c9a06a45_content = inspect.getsource(globals()['preprocess_and_upscale']).splitlines()
cell_c9a06a45_content += inspect.getsource(globals()['convert_note_to_jianpu']).splitlines()
# Extract initial imports and configurations separately from cell c9a06a45
initial_imports_config = [
    "import os",
    "import cv2",
    "import time",
    "import numpy as np",
    "import onnxruntime as ort",
    "from PIL import Image",
    "from music21 import environment, converter, meter, note, stream",
    "from midi2audio import FluidSynth",
    "from IPython.display import Audio, Markdown, display",
    "from tqdm.notebook import tqdm", # Keep tqdm.notebook for now, user can change to tqdm if needed
    "import gradio as gr",
    "import traceback",
    "import sys",
    "import shutil",
    "",
    "# 2. Configure Environment & GPU Detection",
    "env = environment.UserSettings()",
    "env['lilypondPath'] = '/usr/bin/lilypond'",
    "cv2.setUseOptimized(True)",
    "",
    "# Detect if GPU is available for ONNX (optional for app.py, depends on environment)",
    "# HAS_GPU = ort.get_device() == 'GPU'",
    "# if HAS_GPU:\n#     print('🚀 GPU detected! Enabling CUDA acceleration for OMR engine.')\n# else:\n#     print('ℹ️ GPU not detected. Using CPU mode.')\n",
    "# 3. Refined Preprocessing (Optimized for Colab OpenCV)"
]

cell_153275bf_content = inspect.getsource(globals()['process_sheet_music']).splitlines()
cell_gradio_ui_cell_content = inspect.getsource(globals()['ui_wrapper']).splitlines()

# Combine content, handling duplicates and ensuring correct order
app_py_content = []

# Add unique imports and initial configs (excluding the print statements at the end of c9a06a45)
for line in initial_imports_config:
    if line not in app_py_content: # Avoid duplicate general imports
        app_py_content.append(line)

# Add the Keras compatibility line (from 9496033d, but gradio_ui_cell also has it)
# Ensure it's placed before any keras/tf-keras related imports or calls.
if "os.environ['TF_USE_LEGACY_KERAS'] = '1'" not in app_py_content:
    app_py_content.append("os.environ['TF_USE_LEGACY_KERAS'] = '1'")

# Add utility functions
# Skip lines that are already part of initial_imports_config or are print/comment lines.
for line in inspect.getsource(globals()['preprocess_and_upscale']).splitlines():
    if not any(keyword in line for keyword in ['import ', 'from ', 'print(', '#', 'HAS_GPU']) and line.strip() != '':
        app_py_content.append(line)
app_py_content.append("") # Add an empty line for separation

for line in inspect.getsource(globals()['convert_note_to_jianpu']).splitlines():
    if not any(keyword in line for keyword in ['import ', 'from ', 'print(', '#', 'HAS_GPU']) and line.strip() != '':
        app_py_content.append(line)
app_py_content.append("") # Add an empty line for separation

# Add main processing pipeline (process_sheet_music)
for line in cell_153275bf_content:
    # Filter out initial imports as they are already handled
    if not any(keyword in line for keyword in ['import ', 'from ']) and line.strip() != '':
        app_py_content.append(line)
app_py_content.append("") # Add an empty line for separation

# Add Gradio UI setup
# Filter out initial imports and the os.environ line as they are already handled
for line in cell_gradio_ui_cell_content:
    if not any(keyword in line for keyword in ['import ', 'from ', "os.environ['TF_USE_LEGACY_KERAS'] = '1'"]) and line.strip() != '':
        app_py_content.append(line)

# Write to app.py
app_py_filename = 'app.py'
with open(app_py_filename, 'w') as f:
    for line in app_py_content:
        f.write(line + '\n')

print(f"✅ Successfully created '{app_py_filename}' for deployment.")
print("Please review its content to ensure all necessary parts are included and adjust any `tqdm.notebook` to `tqdm` if not running in a notebook environment.")

# Optional: Display the content of app.py
# with open(app_py_filename, 'r') as f:
#     print('\n--- Content of app.py ---')
#     print(f.read())


✅ Successfully created 'app.py' for deployment.
Please review its content to ensure all necessary parts are included and adjust any `tqdm.notebook` to `tqdm` if not running in a notebook environment.
